# Ingest — Azure AI Foundry

Two sources, because cost and usage live in different systems:

| | |
|---|---|
| **Cost Management Query API** | spend and usage quantity, daily, by meter |
| **Azure Monitor Metrics** | token counts and provisioned-capacity utilisation |

Both tables are **optional**. Load neither and the Foundry page is simply empty, which is a
supported state. Load spend alone and you get cost, cost per million tokens and the input/output
split; the metrics table adds PTU utilisation.

**Three things that will otherwise cost you an afternoon**, all verified against a live tenant on
2026-08-07:

- The service is called **`Foundry Models`**, not "Azure OpenAI". Most documentation says the latter.
  Filtering on it returns zero rows, which looks exactly like having no Foundry spend.
- The Cost Management dimension is **`Meter`**, not `MeterName`. The API rejects `MeterName` outright
  and lists the valid dimensions in the 400 body.
- Azure Monitor metric names have changed: **`InputTokens`**, **`OutputTokens`**,
  **`ProvisionedUtilization`** — older material says `ProcessedPromptTokens`, `GeneratedTokens` and
  `AzureOpenAIProvisionedManagedUtilizationV2`. An `OpenAI`-kind resource may still publish the old
  names and an `AIServices`-kind one the new, so this notebook asks the resource which it has.
  Requesting a metric a resource does not publish returns a well-formed series of **zeros** — which
  reads as an idle deployment rather than a wrong name.

**Copilot Studio pay-as-you-go appears in this data too**, as `Pay As You Go Copilot Credit` at
exactly $0.01. The Foundry page uses it to reconcile against the Studio rate you typed into a
parameter. When they disagree, the parameter is wrong — otherwise a completely silent error.

Authentication uses the notebook's own managed identity. It needs **Cost Management Reader** on the
subscription and **Monitoring Reader** on the AI resources.


## Configure

In [ ]:
SUBSCRIPTION_ID = "00000000-0000-0000-0000-000000000000"

# Days of history. Cost Management keeps far more; 90 is enough to see a trend
# without making the first run slow.
DAYS = 90

# Resource-tag keys to try, in order, for department attribution. Tenants name
# this differently and none of them being present is fine - DepartmentTag is
# optional everywhere it is used.
TAG_KEYS = ["Department", "department", "CostCentre", "CostCenter",
            "Team", "BusinessUnit", "Owner"]

LAKEHOUSE_SPEND  = "azure_ai_spend"
LAKEHOUSE_TOKENS = "azure_ai_tokens"


In [ ]:
import json
import requests
from datetime import date, timedelta
from pyspark.sql import functions as F

# The notebook's managed identity, rather than a secret to rotate.
TOKEN = notebookutils.credentials.getToken("https://management.azure.com/")
HEAD  = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
ARM   = "https://management.azure.com"


def arm_get(url, **params):
    r = requests.get(url, headers=HEAD, params=params, timeout=90)
    r.raise_for_status()
    return r.json()


## 1. Spend, from Cost Management

In [ ]:
end   = date.today()
start = end - timedelta(days=DAYS)

body = {
    "type": "ActualCost",
    "timeframe": "Custom",
    "timePeriod": {"from": str(start), "to": str(end)},
    "dataset": {
        "granularity": "Daily",
        "aggregation": {
            "totalCost": {"name": "Cost", "function": "Sum"},
            "totalQty":  {"name": "UsageQuantity", "function": "Sum"},
        },
        # 'Meter', NOT 'MeterName'.
        "grouping": [{"type": "Dimension", "name": d} for d in
                     ["Meter", "MeterCategory", "ResourceId", "ServiceName"]],
        "filter": {"dimensions": {
            "name": "ServiceName",
            "operator": "In",
            # 'Foundry Models' is the real one. The others are included so a
            # tenant on older SKUs still returns something.
            "values": ["Foundry Models", "Microsoft Copilot Studio",
                       "Cognitive Services", "Azure Machine Learning",
                       "Azure OpenAI"],
        }},
    },
}

resp = requests.post(
    f"{ARM}/subscriptions/{SUBSCRIPTION_ID}/providers/Microsoft.CostManagement"
    f"/query?api-version=2025-03-01",
    headers=HEAD, data=json.dumps(body), timeout=180)
resp.raise_for_status()
p    = resp.json()["properties"]
cols = [c["name"] for c in p["columns"]]
rows = p["rows"]
print(f"{len(rows):,} cost rows")


In [ ]:
# Resource tags, fetched separately and joined on resource id. Grouping the
# cost query on a TagKey would be tidier but changes a query that is verified
# working.
tags = {}
res = arm_get(f"{ARM}/subscriptions/{SUBSCRIPTION_ID}/resources",
              **{"api-version": "2021-04-01"})
for r in res.get("value", []):
    t = r.get("tags") or {}
    for k in TAG_KEYS:
        if k in t:
            tags[r["id"].lower()] = t[k]
            break
print(f"{len(tags):,} resources carry a department tag")


In [ ]:
def col(r, n):
    return r[cols.index(n)] if n in cols else None

recs = []
for r in rows:
    rid   = str(col(r, "ResourceId") or "")
    parts = rid.split("/")
    rg    = parts[parts.index("resourcegroups") + 1] if "resourcegroups" in parts else ""
    d     = str(col(r, "UsageDate") or "")
    iso   = f"{d[:4]}-{d[4:6]}-{d[6:]}" if len(d) == 8 else d
    recs.append({
        "UsageDate":     iso,
        "ServiceName":   col(r, "ServiceName"),
        "MeterCategory": col(r, "MeterCategory"),
        "Meter":         col(r, "Meter"),
        "ResourceId":    rid,
        "ResourceName":  parts[-1] if parts else "",
        "ResourceGroup": rg,
        "Cost":          float(col(r, "Cost") or 0),
        "UsageQuantity": float(col(r, "UsageQuantity") or 0),
        "Currency":      col(r, "Currency"),
        "DepartmentTag": tags.get(rid.lower(), ""),
    })

if recs:
    (spark.createDataFrame(recs)
          .withColumn("UsageDate", F.to_date("UsageDate"))
          .write.mode("overwrite").format("delta")
          .saveAsTable(LAKEHOUSE_SPEND))
    print(f"wrote {LAKEHOUSE_SPEND}: {len(recs):,} rows")
else:
    print("no cost rows - check the subscription id and Cost Management Reader")


## 2. Tokens and provisioned utilisation, from Azure Monitor

Optional. Skip this cell if you have no provisioned deployment — the Foundry page says so rather than showing zero.

In [ ]:
WANTED = ["InputTokens", "OutputTokens", "TotalTokens", "TotalCalls",
          "ModelRequests", "ProvisionedUtilization",
          # older names, still emitted by OpenAI-kind resources
          "ProcessedPromptTokens", "GeneratedTokens", "AzureOpenAIRequests",
          "AzureOpenAIProvisionedManagedUtilizationV2"]

accounts = arm_get(
    f"{ARM}/subscriptions/{SUBSCRIPTION_ID}/providers"
    f"/Microsoft.CognitiveServices/accounts",
    **{"api-version": "2023-05-01"}).get("value", [])

points = []
for a in accounts:
    # Ask the resource what it publishes rather than assuming. Requesting a
    # metric it does not have returns zeros, not an error.
    defs = arm_get(f"{ARM}{a['id']}/providers/microsoft.insights/metricDefinitions",
                   **{"api-version": "2018-01-01"}).get("value", [])
    available = {d["name"]["value"] for d in defs}
    ask = [m for m in WANTED if m in available]
    if not ask:
        print(f"  {a['name']}: publishes none of the known token metrics")
        continue
    print(f"  {a['name']}: {', '.join(ask)}")

    md = arm_get(f"{ARM}{a['id']}/providers/microsoft.insights/metrics",
                 **{"api-version": "2024-02-01",
                    "metricnames": ",".join(ask),
                    "interval": "P1D",
                    "timespan": f"{start}/{end}"})
    for v in md.get("value", []):
        nm = v["name"]["value"]
        for s in v.get("timeseries", []):
            for pt in s.get("data", []):
                val = pt.get("total", pt.get("average"))
                if val:
                    points.append({
                        "Date":          pt["timeStamp"][:10],
                        "ResourceName":  a["name"],
                        "ResourceGroup": a["id"].split("/")[4],
                        "Deployment":    "",
                        "Metric":        nm,
                        "Value":         float(val),
                    })

if points:
    (spark.createDataFrame(points)
          .withColumn("Date", F.to_date("Date"))
          .write.mode("overwrite").format("delta")
          .saveAsTable(LAKEHOUSE_TOKENS))
    print(f"wrote {LAKEHOUSE_TOKENS}: {len(points):,} metric points")
else:
    print("no metric points. If the names above were listed as available, this")
    print("is genuinely no traffic rather than a wrong metric name.")
